# Caso de estudio en Salud — Leucemia Linfoblástica Aguda (ALL)

Versión **local** del notebook de Colab. Cambios respecto al original:

- No hay `google.colab.drive`: los datos están en `data/` dentro del proyecto.
- El split 70/15/15 lo hace `src/prep_data.py` (misma semilla 42) y además reduce
  las imágenes a 180 px, porque los originales son de hasta 5 MP y aquí se entrena en CPU.
- Se ejecuta sobre las 4 clases con `class_mode='categorical'` (el original tenía
  restos de la configuración binaria).

## Comprensión del Negocio (Business Understanding)

**Objetivo:** automatizar la detección de células leucémicas en sangre periférica para asistir el diagnóstico de ALL, clasificando en 4 clases: `Benign`, `[Malignant] Pre-B`, `[Malignant] Pro-B`, `[Malignant] early Pre-B`.

Se construyen y comparan dos clasificadores: una CNN propia y VGG16 preentrenado en ImageNet con fine-tuning.

In [ ]:
import os, sys, json, random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (Conv2D, MaxPooling2D, Flatten, Dense,
                                     GlobalAveragePooling2D, Dropout)
from tensorflow.keras.applications import VGG16
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, os.path.join(ROOT, 'src'))
print('TensorFlow', tf.__version__, '| dispositivos:', tf.config.list_physical_devices())
print('ROOT:', ROOT)

## Comprensión de los Datos (Data Understanding)

In [ ]:
DATA_ORIG = os.path.join(ROOT, 'data', 'Blood cell Cancer [ALL]')
DESTINO   = os.path.join(ROOT, 'data', 'split')

class_names = sorted(d for d in os.listdir(DATA_ORIG) if os.path.isdir(os.path.join(DATA_ORIG, d)))
print('Clases:', class_names)
for cls in class_names:
    print(f'  {cls:26s} {len(os.listdir(os.path.join(DATA_ORIG, cls))):5d} imágenes')

In [ ]:
# Una imagen aleatoria de cada clase
plt.figure(figsize=(14, 4))
for idx, cls in enumerate(class_names):
    d = os.path.join(DATA_ORIG, cls)
    img = mpimg.imread(os.path.join(d, random.choice(os.listdir(d))))
    plt.subplot(1, len(class_names), idx + 1)
    plt.imshow(img); plt.title(f'Clase: {cls}', fontsize=9); plt.axis('off')
plt.tight_layout(); plt.show()

## Preparación de los Datos (Data Preparation)

Split 70/15/15 por clase con semilla 42. Si `data/split` ya existe (creado por `python src/prep_data.py`), esta celda no hace nada.

In [ ]:
if not os.path.isdir(DESTINO):
    !python {os.path.join(ROOT, 'src', 'prep_data.py')}
else:
    for split in ['train', 'val', 'test']:
        total = sum(len(os.listdir(os.path.join(DESTINO, split, c))) for c in class_names)
        print(f'{split:5s}: {total} imágenes')

In [ ]:
IMG_WIDTH, IMG_HEIGHT = 150, 150
BATCH_SIZE = 32
EPOCHS = 15

datagen = ImageDataGenerator(rescale=1./255)

train_generator = datagen.flow_from_directory(
    os.path.join(DESTINO, 'train'), target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=True)

val_generator = datagen.flow_from_directory(
    os.path.join(DESTINO, 'val'), target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)

test_generator = datagen.flow_from_directory(
    os.path.join(DESTINO, 'test'), target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)

class_names = list(train_generator.class_indices.keys())
print(train_generator.class_indices)

In [ ]:
images, labels = next(train_generator)
plt.figure(figsize=(10, 10))
for i in range(9):
    plt.subplot(3, 3, i + 1)
    plt.imshow(images[i])
    plt.title(class_names[labels[i].argmax()], fontsize=9)
    plt.axis('off')
plt.show()

## Modelado (Modeling)

### Modelo 1 — CNN propia

In [ ]:
model_cnn = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
    MaxPooling2D(2, 2),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Flatten(),
    Dense(512, activation='relu'),
    Dense(train_generator.num_classes, activation='softmax'),
])

model_cnn.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model_cnn.summary()

In [ ]:
MODELS = os.path.join(ROOT, 'models'); os.makedirs(MODELS, exist_ok=True)

callbacks_cnn = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    ModelCheckpoint(os.path.join(MODELS, 'best_model_scratch.keras'),
                    monitor='val_loss', save_best_only=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6),
]

history_cnn = model_cnn.fit(train_generator, epochs=EPOCHS,
                            validation_data=val_generator, callbacks=callbacks_cnn)

In [ ]:
train_loss_cnn = history_cnn.history['loss']
val_loss_cnn = history_cnn.history['val_loss']
epochs_cnn = range(1, len(train_loss_cnn) + 1)

plt.figure(figsize=(10, 6))
plt.plot(epochs_cnn, train_loss_cnn, marker='o', label='Pérdida Entrenamiento')
plt.plot(epochs_cnn, val_loss_cnn, marker='o', label='Pérdida validación')
plt.title('Pérdida en entrenamiento y validación -- CNN Propia')
plt.xlabel('Epoch'); plt.ylabel('Pérdida'); plt.legend(); plt.show()

## Evaluación (Evaluation) — CNN propia

In [ ]:
def evaluar(model, generator, titulo):
    generator.reset()
    predictions = model.predict(generator)
    predicted_classes = np.argmax(predictions, axis=1)
    true_classes = generator.classes

    acc = accuracy_score(true_classes, predicted_classes)
    print(f'Accuracy en test: {acc:.4f}\n')
    print(classification_report(true_classes, predicted_classes,
                                target_names=class_names, digits=4))

    mtx = confusion_matrix(true_classes, predicted_classes)
    fig, ax = plt.subplots(figsize=(14, 8))
    sns.heatmap(mtx, cmap='Blues', xticklabels=class_names, yticklabels=class_names,
                annot=True, annot_kws={'size': 10}, fmt='d', linewidths=.1)
    plt.grid(False); plt.yticks(rotation=0)
    plt.xlabel('Predicted labels'); plt.ylabel('True labels')
    plt.title(f'{titulo} (accuracy = {acc:.3f})'); plt.show()
    return acc

accuracy_cnn = evaluar(model_cnn, test_generator, 'Matriz de confusión -- CNN Propia')

### Modelo 2 — VGG16 preentrenado con fine-tuning

Fase 1: base congelada, sólo la cabeza nueva. Fase 2: se descongela `block5` y se reentrena con learning rate bajo.

In [ ]:
base_model = VGG16(weights='imagenet', include_top=False,
                   input_shape=(IMG_HEIGHT, IMG_WIDTH, 3))
base_model.trainable = False

x = GlobalAveragePooling2D()(base_model.output)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
output = Dense(train_generator.num_classes, activation='softmax')(x)

model_vgg16 = Model(inputs=base_model.input, outputs=output)
model_vgg16.compile(optimizer=Adam(learning_rate=1e-4),
                    loss='categorical_crossentropy', metrics=['accuracy'])
model_vgg16.summary()

In [ ]:
EPOCHS_INITIAL = 15
EPOCHS_FINETUNE = 5

callbacks_vgg = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ModelCheckpoint(os.path.join(MODELS, 'best_model_vgg16.keras'),
                    monitor='val_loss', save_best_only=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6),
]

history_pretrained = model_vgg16.fit(train_generator, epochs=EPOCHS_INITIAL,
                                     validation_data=val_generator, callbacks=callbacks_vgg)

In [ ]:
# Fase 2: fine-tuning del último bloque convolucional
base_model.trainable = True
for layer in base_model.layers:
    layer.trainable = layer.name.startswith('block5')

model_vgg16.compile(optimizer=Adam(learning_rate=1e-5),
                    loss='categorical_crossentropy', metrics=['accuracy'])

history_finetune = model_vgg16.fit(train_generator, epochs=EPOCHS_FINETUNE,
                                   validation_data=val_generator, callbacks=callbacks_vgg)

In [ ]:
train_loss = history_pretrained.history['loss'] + history_finetune.history['loss']
val_loss = history_pretrained.history['val_loss'] + history_finetune.history['val_loss']
epochs = range(1, len(train_loss) + 1)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(epochs, train_loss, marker='o', label='Pérdida Entrenamiento')
ax.plot(epochs, val_loss, marker='o', label='Pérdida validación')
ax.axvline(len(history_pretrained.history['loss']) + 0.5, ls='--', c='gray', lw=1)
plt.title('Pérdida en entrenamiento y validación -- VGG16 (| = inicio del fine-tuning)')
plt.xlabel('Época'); plt.ylabel('Pérdida'); plt.legend(); plt.show()

In [ ]:
accuracy_vgg16 = evaluar(model_vgg16, test_generator, 'Matriz de confusión -- VGG16 fine-tuned')

print(f'\nComparación final:')
print(f'  CNN propia : {accuracy_cnn:.4f}')
print(f'  VGG16 (FT) : {accuracy_vgg16:.4f}')

## Despliegue (Deployment)

Carga del mejor modelo guardado y predicción sobre una imagen nueva. A diferencia del original (que devolvía `'ALL'`/`'Sano'` con un umbral binario), aquí se devuelve la clase de las 4 con su probabilidad.

In [ ]:
def predecir(image_path, model_file='best_model_scratch.keras'):
    model = tf.keras.models.load_model(os.path.join(MODELS, model_file))
    img = tf.keras.preprocessing.image.load_img(image_path, target_size=(IMG_HEIGHT, IMG_WIDTH))
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0) / 255.0
    probs = model.predict(img_array, verbose=0)[0]
    idx = int(np.argmax(probs))
    return class_names[idx], float(probs[idx]), dict(zip(class_names, probs.round(4).tolist()))

# Ejemplo con una imagen del conjunto de test
cls_demo = class_names[0]
demo_dir = os.path.join(DESTINO, 'test', cls_demo)
demo_img = os.path.join(demo_dir, sorted(os.listdir(demo_dir))[0])

pred, prob, todas = predecir(demo_img)
print(f'Imagen   : {os.path.basename(demo_img)} (real: {cls_demo})')
print(f'Predicción: {pred}  ({prob:.1%})')
print('Probabilidades:', todas)

## Conclusiones

_(completar con los resultados obtenidos al ejecutar el notebook)_

- Accuracy de la CNN propia en test: …
- Accuracy de VGG16 con fine-tuning en test: …
- Clases más confundidas según la matriz de confusión: …